In [ ]:
!pip install -q neo4j

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.5/331.5 kB 6.8 MB/s eta 0:00:00


In [ ]:
import neo4j

print("Neo4j Python Driver Version:", neo4j.__version__)

Neo4j Python Driver Version: 6.3.1


1.เริ่มติดต่อใช้งาน Neo4j online server

In [ ]:
from getpass import getpass
from neo4j import GraphDatabase

URI = "neo4j+s://b9b3be4a.databases.neo4j.io"
USERNAME = "b9b3be4a"
PASSWORD = getpass("Password: ")

driver = GraphDatabase.driver(
    URI,
    auth=(USERNAME, PASSWORD)
)

driver.verify_connectivity()

print("Connected successfully")

Password: ··········
Connected successfully


หา home database

In [ ]:
result = driver.execute_query(
    "SHOW HOME DATABASE"
)

for record in result.records:
    print(record.data())

{'name': 'b9b3be4a', 'type': 'standard', 'aliases': [], 'access': 'read-write', 'address': 'p-mt-3b814dd7cab7-16-0136.production-orch-0068.neo4j.io:7687', 'role': 'primary', 'writer': True, 'requestedStatus': 'online', 'currentStatus': 'online', 'statusMessage': '', 'constituents': []}


เก็บชื่อ database

In [ ]:
DATABASE = result.records[0]["name"]

print("Database =", DATABASE)

Database = b9b3be4a


หลังจากนั้น Query ทุกคำสั่งใช้:

In [ ]:
result = driver.execute_query(
    """
    RETURN 'Hello Neo4j' AS message
    """,
    database_=DATABASE
)

print(result.records[0]["message"])

Hello Neo4j


ทดสอบใช้งานฐานข้อมูลจริง

In [ ]:
result = driver.execute_query(
    """
    MATCH (n)
    RETURN count(n) AS total_nodes
    """,
    database_=DATABASE
)

print("จำนวน Node =", result.records[0]["total_nodes"])

จำนวน Node = 20


สร้าง Node

In [ ]:
result = driver.execute_query(
    """
    CREATE (u:User {
        username: 'Alice'
    })

    RETURN u
    """,
    database_=DATABASE
)

print(result.records[0]["u"])

ConstraintError: {neo4j_code: Neo.ClientError.Schema.ConstraintValidationFailed} {message: Node(8) already exists with label `User` and property `username` = 'Alice'} {gql_status: 22N79} {gql_status_description: error: data exception - property uniqueness constraint violated. Property uniqueness constraint violated: Node(8) already exists with label `User` and property `username` = 'Alice'.}

ตรวจสอบผลลัพธ์

In [ ]:
result = driver.execute_query(
    """
    MATCH (u:User)
    RETURN
        u.username AS username
    """,
    database_=DATABASE,
)

for record in result.records:
    print(record.data())

{'username': 'Ivy'}
{'username': 'Jack'}
{'username': 'Alice'}
{'username': 'Bob'}
{'username': 'Charlie'}
{'username': 'David'}
{'username': 'Emma'}
{'username': 'Frank'}
{'username': 'Grace'}
{'username': 'Henry'}


ทดลอง where

In [ ]:
result = driver.execute_query(
    """
    MATCH (u:User)

    WHERE u.username = 'Alice'

    RETURN
        u.username AS name
    """,
    database_=DATABASE
)

for record in result.records:
    print(record.data())

{'name': 'Alice'}


6.สร้าง constrain

In [ ]:
driver.execute_query(
    """
    CREATE CONSTRAINT user_username_unique
    IF NOT EXISTS
    FOR (u:User)
    REQUIRE u.username IS UNIQUE
    """,
    database_=DATABASE,
)

print("User constraint created")

User constraint created


In [ ]:
driver.execute_query(
    """
    CREATE CONSTRAINT game_name_unique
    IF NOT EXISTS

    FOR (g:Game)

    REQUIRE g.game_name IS UNIQUE
    """,
    database_=DATABASE
)

print("Game constraint created")

Game constraint created


*เพิ่ม* User หลายๆคน

In [ ]:
users = [
    "Alice",
    "Bob",
    "Charlie",
    "David",
    "Emma",
    "Frank",
    "Grace",
    "Henry",
    "Ivy",
    "Jack",
]

In [ ]:
result = driver.execute_query(
    """
    UNWIND $users AS username

    MERGE (u:User {
        username: username
    })

    RETURN count(u) AS total
    """,
    users=users,
    database_=DATABASE,
)

print(
    "จำนวน User ที่ประมวลผล =",
    result.records[0]["total"],
)

จำนวน User ที่ประมวลผล = 10


In [ ]:
games = [
    "Elden Ring",
    "Cyberpunk 2077",
    "Minecraft",
    "Valorant",
    "Genshin Impact",
    "Honkai: Star Rail",
    "The Witcher 3",
    "GTA V",
    "Apex Legends",
    "Project Zomboid",
]

In [ ]:
result = driver.execute_query(
    """
    UNWIND $games AS game_name

    MERGE (g:Game {
        game_name: game_name
    })

    RETURN count(g) AS total
    """,
    games=games,
    database_=DATABASE,
)

print(
    "จำนวน Game ที่ประมวลผล =",
    result.records[0]["total"],
)

จำนวน Game ที่ประมวลผล = 10


ดู User และ Game ทั้งหมด

In [ ]:
result = driver.execute_query(
    """
    MATCH (n)

    WHERE n:User OR n:Game

    RETURN
        labels(n) AS type,
        properties(n) AS data
    """,
    database_=DATABASE,
)

for record in result.records:
    print(record.data())

{'type': ['User'], 'data': {'username': 'Ivy'}}
{'type': ['User'], 'data': {'username': 'Jack'}}
{'type': ['Game'], 'data': {'game_name': 'Elden Ring'}}
{'type': ['Game'], 'data': {'game_name': 'Cyberpunk 2077'}}
{'type': ['Game'], 'data': {'game_name': 'Minecraft'}}
{'type': ['Game'], 'data': {'game_name': 'Valorant'}}
{'type': ['Game'], 'data': {'game_name': 'Genshin Impact'}}
{'type': ['Game'], 'data': {'game_name': 'Honkai: Star Rail'}}
{'type': ['User'], 'data': {'username': 'Alice'}}
{'type': ['User'], 'data': {'username': 'Bob'}}
{'type': ['User'], 'data': {'username': 'Charlie'}}
{'type': ['User'], 'data': {'username': 'David'}}
{'type': ['User'], 'data': {'username': 'Emma'}}
{'type': ['User'], 'data': {'username': 'Frank'}}
{'type': ['User'], 'data': {'username': 'Grace'}}
{'type': ['User'], 'data': {'username': 'Henry'}}
{'type': ['Game'], 'data': {'game_name': 'The Witcher 3'}}
{'type': ['Game'], 'data': {'game_name': 'GTA V'}}
{'type': ['Game'], 'data': {'game_name': 'Apex

Step 10 — สร้าง Relationship FRIEND_OF

In [ ]:
friendships = [
    {"user1": "Alice", "user2": "Bob"},
    {"user1": "Alice", "user2": "Charlie"},
    {"user1": "Alice", "user2": "David"},
    {"user1": "Bob", "user2": "Charlie"},
    {"user1": "Bob", "user2": "Emma"},
    {"user1": "Charlie", "user2": "David"},
    {"user1": "Charlie", "user2": "Emma"},
]

In [ ]:
driver.execute_query(
    """
    UNWIND $friendships AS row

    MATCH (a:User {username: row.user1})
    MATCH (b:User {username: row.user2})

    MERGE (a)-[:FRIEND_OF]->(b)
    """,
    friendships=friendships,
    database_=DATABASE,
)

print("สร้าง FRIEND_OF สำเร็จ")

สร้าง FRIEND_OF สำเร็จ


Step 11 — Query ความสัมพันธ์

In [ ]:
result = driver.execute_query(
    """
    MATCH (u:User {username: $target_user})-[:FRIEND_OF]-(friend:User)
    RETURN friend.username AS friend
    """,
    target_user="Alice",
    database_=DATABASE,
)

for record in result.records:
    print(record["friend"])

Bob
Charlie
David


Step 12 — สร้าง Like Relationship

กำหนดการชอบ

In [ ]:
likes = [
    ("Alice", "Elden Ring"),
    ("Alice", "The Witcher 3"),
    ("Alice", "Cyberpunk 2077"),
    ("Bob", "Valorant"),
    ("Bob", "Apex Legends"),
    ("Bob", "GTA V"),
    ("Charlie", "Genshin Impact"),
    ("Charlie", "Honkai: Star Rail"),
    ("Charlie", "Minecraft"),
    ("David", "Elden Ring"),
    ("David", "Project Zomboid"),
    ("David", "Cyberpunk 2077"),
    ("Emma", "Minecraft"),
    ("Emma", "Project Zomboid"),
    ("Emma", "Genshin Impact"),
    ("Frank", "Valorant"),
    ("Frank", "Apex Legends"),
    ("Frank", "The Witcher 3"),
    ("Grace", "Honkai: Star Rail"),
    ("Grace", "Genshin Impact"),
    ("Grace", "Elden Ring"),
    ("Henry", "GTA V"),
    ("Henry", "Cyberpunk 2077"),
    ("Henry", "Project Zomboid"),
    ("Ivy", "Minecraft"),
    ("Ivy", "Honkai: Star Rail"),
    ("Ivy", "Apex Legends"),
    ("Jack", "GTA V"),
    ("Jack", "Valorant"),
    ("Jack", "The Witcher 3"),
]

สร้าง Relationship

In [ ]:
driver.execute_query(
    """
    UNWIND $likes AS pair

    MATCH
        (u:User {username: pair[0]}),
        (g:Game {game_name: pair[1]})

    MERGE
        (u)-[r:LIKES]->(g)
    """,
    likes=likes,
    database_=DATABASE,
)

print("สร้าง LIKES สำเร็จ")

สร้าง LIKES สำเร็จ


Step 13 — ดูว่าใครชอบอะไร

In [ ]:
result = driver.execute_query(
    """
    MATCH
        (u:User)
        -[r:LIKES]->
        (g:Game)

    RETURN
        u.username AS username,
        g.game_name AS game

    ORDER BY username, game
    """,
    database_=DATABASE,
)

for record in result.records:
    print(
        record["username"],
        "->",
        record["game"],
    )

Alice -> Cyberpunk 2077
Alice -> Elden Ring
Alice -> The Witcher 3
Bob -> Apex Legends
Bob -> GTA V
Bob -> Valorant
Charlie -> Genshin Impact
Charlie -> Honkai: Star Rail
Charlie -> Minecraft
David -> Cyberpunk 2077
David -> Elden Ring
David -> Project Zomboid
Emma -> Genshin Impact
Emma -> Minecraft
Emma -> Project Zomboid
Frank -> Apex Legends
Frank -> The Witcher 3
Frank -> Valorant
Grace -> Elden Ring
Grace -> Genshin Impact
Grace -> Honkai: Star Rail
Henry -> Cyberpunk 2077
Henry -> GTA V
Henry -> Project Zomboid
Ivy -> Apex Legends
Ivy -> Honkai: Star Rail
Ivy -> Minecraft
Jack -> GTA V
Jack -> The Witcher 3
Jack -> Valorant


Step 14 — นำผลลัพธ์มาแสดงเป็นตาราง Pandas

In [ ]:
import pandas as pd

In [ ]:
result = driver.execute_query(
    """
    MATCH
        (u:User)
        -[r:LIKES]->
        (g:Game)

    RETURN
        u.username AS username,
        g.game_name AS game

    ORDER BY username, game
    """,
    database_=DATABASE,
)

data = [record.data() for record in result.records]

df = pd.DataFrame(data)

df

,username,game
0,Alice,Cyberpunk 2077
1,Alice,Elden Ring
2,Alice,The Witcher 3
3,Bob,Apex Legends
4,Bob,GTA V
5,Bob,Valorant
6,Charlie,Genshin Impact
7,Charlie,Honkai: Star Rail
8,Charlie,Minecraft
9,David,Cyberpunk 2077


Step 15 — Traversal หลายทอด

โจทย์:

เพื่อนของ Alice ยืมหนังสืออะไรบ้าง?

In [ ]:
result = driver.execute_query(
    """
    MATCH
        (me:User {username: $username})
        -[:FRIEND_OF]-
        (friend:User)
        -[:LIKES]->
        (game:Game)

    RETURN
        me.username AS me,
        friend.username AS friend,
        game.game_name AS game

    ORDER BY friend, game
    """,
    username="Alice",  # ระบุชื่อผู้ใช้ที่ต้องการค้นหา
    database_=DATABASE,
)

for record in result.records:
    print(
        record["me"],
        "-> เพื่อน:",
        record["friend"],
        "-> ชอบเกม:",
        record["game"],
    )

Alice -> เพื่อน: Bob -> ชอบเกม: Apex Legends
Alice -> เพื่อน: Bob -> ชอบเกม: GTA V
Alice -> เพื่อน: Bob -> ชอบเกม: Valorant
Alice -> เพื่อน: Charlie -> ชอบเกม: Genshin Impact
Alice -> เพื่อน: Charlie -> ชอบเกม: Honkai: Star Rail
Alice -> เพื่อน: Charlie -> ชอบเกม: Minecraft
Alice -> เพื่อน: David -> ชอบเกม: Cyberpunk 2077
Alice -> เพื่อน: David -> ชอบเกม: Elden Ring
Alice -> เพื่อน: David -> ชอบเกม: Project Zomboid


Step 16 — Aggregation

โจทย์:

นักเรียนแต่ละคนชอบเกมกี่เกม

In [ ]:
result = driver.execute_query(
    """
    MATCH
        (u:User)
        -[:LIKES]->
        (g:Game)

    RETURN
        u.username AS username,
        count(g) AS total_games

    ORDER BY total_games DESC
    """,
    database_=DATABASE,
)

for record in result.records:
    print(
        record["username"],
        "=",
        record["total_games"],
        "เกม",
    )

Alice = 3 เกม
David = 3 เกม
Grace = 3 เกม
Henry = 3 เกม
Ivy = 3 เกม
Charlie = 3 เกม
Emma = 3 เกม
Jack = 3 เกม
Bob = 3 เกม
Frank = 3 เกม


Step 17 — เกมไหนได้รับความนิยมมากที่สุด

In [ ]:
result = driver.execute_query(
    """
    MATCH
        (:User)
        -[:LIKES]->
        (g:Game)

    RETURN
        g.game_name AS game,
        count(*) AS like_count

    ORDER BY like_count DESC
    """,
    database_=DATABASE,
)

pd.DataFrame(
    [record.data() for record in result.records]
)

,game,like_count
0,Minecraft,3
1,Honkai: Star Rail,3
2,Apex Legends,3
3,Valorant,3
4,The Witcher 3,3
5,GTA V,3
6,Elden Ring,3
7,Cyberpunk 2077,3
8,Genshin Impact,3
9,Project Zomboid,3


Step 18 — Mini Project: Recommendation System

โจทย์สำคัญที่สุด

แนะนำเกมให้ Alice จากเกมที่เพื่อนชอบ แต่ Alice ยังไม่เคยชอบ/เล่น

In [ ]:
result = driver.execute_query(
    """
    MATCH
        (me:User {username: $username})
        -[:FRIEND_OF]-
        (friend:User)
        -[:LIKES]->
        (game:Game)

    WHERE NOT EXISTS {
        MATCH (me)-[:LIKES]->(game)
    }

    RETURN
        game.game_name AS recommendation,
        count(DISTINCT friend) AS friend_score

    ORDER BY
        friend_score DESC,
        recommendation
    """,
    username="Alice",
    database_=DATABASE,
)

recommend_df = pd.DataFrame(
    [record.data() for record in result.records]
)

recommend_df

,recommendation,friend_score
0,Apex Legends,1
1,GTA V,1
2,Genshin Impact,1
3,Honkai: Star Rail,1
4,Minecraft,1
5,Project Zomboid,1
6,Valorant,1


ทำไม B103 ได้คะแนนสูงกว่า?


```
เพราะ

             Mali
            /
Anan ─ FRIEND
            \
             Somchai

และ

Mali ─────── BORROWED ──┐
                         ↓
               Data Science
                         ↑
Somchai ─── BORROWED ───┘
"""

ดังนั้นมีเพื่อน 2 คน ของ Anan ที่เคยยืม B103

friend_score = 2


```



Step 19 — ทำเป็น Python Function

เปลี่ยน Mini Project ให้ดูเหมือนโปรแกรมจริง

In [ ]:
def recommend_games(username):
    query = """
    MATCH
        (me:User {username: $username})
        -[:FRIEND_OF]-
        (friend:User)
        -[:LIKES]->
        (game:Game)

    WHERE NOT EXISTS {
        MATCH (me)-[:LIKES]->(game)
    }

    RETURN
        game.game_name AS recommendation,
        count(DISTINCT friend) AS score

    ORDER BY
        score DESC,
        recommendation
    """

    result = driver.execute_query(
        query,
        username=username,
        database_=DATABASE,
    )

    return pd.DataFrame([record.data() for record in result.records])

In [ ]:
recommend_games("Emma")

,recommendation,score
0,Apex Legends,1
1,GTA V,1
2,Honkai: Star Rail,1
3,Valorant,1


Step 20 — Function ค้นหานักเรียน

In [ ]:
def find_user(username):
    result = driver.execute_query(
        """
        MATCH (u:User {username: $username})

        RETURN
            u.username AS username
        """,
        username=username,
        database_=DATABASE,
    )

    if len(result.records) == 0:
        return "ไม่พบผู้ใช้"

    return result.records[0].data()

In [ ]:
find_user("Grace")

{'username': 'Grace'}

Step 21 — Function ดูหนังสือที่นักเรียนเคยยืม

In [ ]:
def liked_games(username):
    result = driver.execute_query(
        """
        MATCH
            (u:User {username: $username})
            -[r:LIKES]->
            (g:Game)

        RETURN
            g.game_name AS game

        ORDER BY game
        """,
        username=username,
        database_=DATABASE,
    )

    return pd.DataFrame([record.data() for record in result.records])

In [ ]:
liked_games("Alice")

,game
0,Cyberpunk 2077
1,Elden Ring
2,The Witcher 3


Step 22 — ดู Graph ใน Neo4j Aura

หลังจากรันข้อมูลจาก Colab แล้ว ให้นักเรียนกลับไปที่ Neo4j Aura → Query

รัน


In [ ]:
recommend_games('Alice')

,recommendation,score
0,Apex Legends,1
1,GTA V,1
2,Genshin Impact,1
3,Honkai: Star Rail,1
4,Minecraft,1
5,Project Zomboid,1
6,Valorant,1


In [ ]:
#Step 23 — ตรวจจำนวน Node และ Relationship
result = driver.execute_query(
    """
    MATCH (n)

    OPTIONAL MATCH (n)-[r]->()

    RETURN
        count(DISTINCT n) AS nodes,
        count(r) AS relationships
    """,
    database_=DATABASE
)

print(result.records[0].data())

{'nodes': 20, 'relationships': 37}


In [ ]:
#Step 24 — ปิด Connection เมื่อเรียนเสร็จ

driver.close()

print("Neo4j connection closed")